# 小车 v0.4 听懂文字 (仿真版) · 在 Colab 上运行

让大模型通过工具调用开一台仿真小车, 并在 30 个任务上实测。讲解见 [本站讲义](https://github.com/OGGIoao/roots-ai/blob/master/projects/car/v0.4-text-commands/TUTORIAL.md); Agent 循环本身的原理见知识库 [agent-loop](https://github.com/OGGIoao/roots-ai/blob/master/library/agent/agent-loop/TUTORIAL.md)。

前两步只用 CPU; 第 3 步用本地小模型 (Qwen2.5-1.5B-Instruct, 约 3 GB) 跑评测, 需要 GPU
(菜单 "代码执行程序 → 更改运行时类型" 选 GPU)。Colab 能直接访问 Hugging Face, 不需要镜像。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OGGIoao/roots-ai/blob/master/projects/car/v0.4-text-commands/colab.ipynb)

In [ ]:
# 克隆仓库并安装共享包 (Colab 每次新开会话都要重新跑这一格)
!git clone --depth 1 https://github.com/OGGIoao/roots-ai.git
%cd roots-ai
!pip install -q -e . --no-deps

## 1. 先看循环长什么样, 再看规则控制器 (都不用大模型)

第一格用一个假模型把整段对话打印出来; 第二格是同样的工具和任务、大脑换成几十行 if-else 的对照组。

In [ ]:
!python library/agent/agent-loop/demo_minimal.py --seed 1
!python projects/car/v0.4-text-commands/baseline.py

## 2. 小模型评测 (6 个任务, 快速确认能跑通)

讲义里的数字来自每组 30 个任务 (`--n 30`); 这里先各跑 6 个。`loop` 是让模型自己一步步开车, `skill` 是只让它选目标 (讲义第 6 节)。
每局会打印: 成功与否、离目标多远、撞了几次、调用了几次工具。

In [ ]:
!cd projects/car/v0.4-text-commands && python evaluate.py --backend local:Qwen/Qwen2.5-1.5B-Instruct --mode loop --n 6
!cd projects/car/v0.4-text-commands && python evaluate.py --backend local:Qwen/Qwen2.5-1.5B-Instruct --mode skill --n 6

## 3. 看每一局是怎么失败的

把每一局归到一类结局 (讲义第 5 节的表)。

In [ ]:
!cd projects/car/v0.4-text-commands && python analyze.py

## 练习

练习文件在左侧"文件"面板里 (`roots-ai/<本章所在目录>/exercises/`), 双击打开就能编辑, 改完按 Ctrl+S 保存, 再运行下面的测试。
刚克隆下来时测试会失败, 这是正常的: 报 `NotImplementedError` (还没写), 或者 `RuntimeError: Error(s) in loading state_dict` (还没写完, 参数对不上)。等你把 TODO 补完再跑。

> **注意**: Colab 会话断开后, 你改过的文件会丢失。想保留, 就把练习文件下载下来 (右键 → 下载), 或者 fork 本仓库后把上面的克隆地址换成你自己的。

In [ ]:
!cd library/agent/agent-loop/exercises && python -m pytest test_ex1.py -q
!cd projects/car/v0.4-text-commands/exercises && python -m pytest test_ex3.py -q